# Notebook 01 - Set up and explore PTB-XL
**Thesis:** Development of an Explainable Deep Learning Model for 12-Lead Electrocardiogram Abnormality Classification (Chapter 3, Sections 3.2.1-3.2.2)

What this notebook does:
1. Connects Google Drive and installs the libraries.
2. Downloads PTB-XL (open, about 1.7 GB) and PTB-XL+ (open, about 0.7 GB) from PhysioNet into your Drive, once.
3. Describes the dataset for **Chapter 4, Section 4.2** (Table 4.1 and Figure 4.1).

Before you start: **Runtime > Change runtime type > T4 GPU > Save.** (This notebook does not need the GPU, but later ones do.)

In [ ]:
# STEP 0 - Setup. Run this cell first, every time you open the notebook.
import os, sys, glob, json, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"   # the branch of your GitHub repository to use

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                       # click "Connect to Google Drive" and allow
    PROJECT = "/content/drive/MyDrive/FUTA-MTECH"        # everything you keep lives here
    REPO = "/content/FUTA-MTECH-PROJECT"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/Scientist47/FUTA-MTECH-PROJECT.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip -q install wfdb neurokit2 shap captum interpret-core
else:                                                    # running on your own computer
    PROJECT = os.environ.get("FUTA_PROJECT", os.path.abspath(".."))
    REPO = os.environ.get("FUTA_REPO", os.path.abspath(".."))

sys.path.insert(0, REPO)
os.chdir(REPO)
DATA = f"{PROJECT}/data"            # downloaded and processed data
RESULTS = f"{PROJECT}/results"      # tables, figures and trained models for Chapter 4
for d in (DATA, RESULTS, f"{RESULTS}/tables", f"{RESULTS}/figures", f"{RESULTS}/models"):
    os.makedirs(d, exist_ok=True)

# QUICK_TEST = True runs everything with tiny settings, to check the notebook works.
# Leave it False for your real thesis results.
QUICK_TEST = os.environ.get("FUTA_QUICK_TEST") == "1"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
print("GPU available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only - see the guide, Step 1.3)")
print("Project folder:", PROJECT)

## Step 1.1 - Download PTB-XL and PTB-XL+ (first time only, 10-30 minutes)
The files are saved to `MyDrive/FUTA-MTECH/data/`, so you only download once. Each later session copies the zip from Drive to the fast Colab disk and unzips it (about 2 minutes).

In [ ]:
def get_dataset(name, url, marker):
    """Download a PhysioNet zip into Drive once, unzip it to the Colab disk, return its folder."""
    zip_path = f"{DATA}/{name}.zip"
    local = f"/content/{name}" if IN_COLAB else f"{DATA}/{name}"
    found = glob.glob(f"{local}/**/{marker}", recursive=True)
    if found:
        return os.path.dirname(found[0])
    if not os.path.exists(zip_path):
        print(f"Downloading {name} ... (this can take a while)")
        !wget -q --show-progress -O "{zip_path}" "{url}"
    print("Unzipping ...")
    !unzip -q -o "{zip_path}" -d "{local}"
    return os.path.dirname(glob.glob(f"{local}/**/{marker}", recursive=True)[0])

PTBXL = os.environ.get("FUTA_PTBXL") or get_dataset(
    "ptb-xl", "https://physionet.org/content/ptb-xl/get-zip/1.0.3/", "ptbxl_database.csv")
print("PTB-XL folder:", PTBXL)

In [ ]:
# PTB-XL+ (measured features for the EBM comparator, and fiducial points).
# Skip this cell in a quick test; it is needed in notebooks 03 and 04.
if not QUICK_TEST:
    PTBXL_PLUS = get_dataset("ptb-xl-plus", "https://physionet.org/content/ptb-xl-plus/get-zip/1.0.1/",
                             "12sl_features.csv")
    PTBXL_PLUS = os.path.dirname(PTBXL_PLUS)        # the folder that contains features/
    print("PTB-XL+ folder:", PTBXL_PLUS)
    print(os.listdir(PTBXL_PLUS))

In [ ]:
# Remember where the data are, for the other notebooks
paths = {"PTBXL": PTBXL}
if not QUICK_TEST:
    paths["PTBXL_PLUS"] = PTBXL_PLUS
json.dump(paths, open(f"{PROJECT}/paths.json", "w"), indent=1)
paths

## Step 1.2 - Load the metadata and the five superclass labels
`load_metadata` reads `ptbxl_database.csv`, maps each record's SCP codes to NORM, MI, STTC, CD and HYP with `scp_statements.csv`, and drops records with no diagnostic superclass (as in the benchmark of Strodthoff et al., 2021).

In [ ]:
from preprocessing.ptbxl import load_metadata, SUPERCLASSES
meta_all = load_metadata(PTBXL, drop_unlabelled=False)
meta = meta_all[meta_all["n_superclasses"] > 0]
print(f"All records: {len(meta_all):,}   from {meta_all.patient_id.nunique():,} patients")
print(f"Records with at least one superclass: {len(meta):,}")
meta[["patient_id", "strat_fold", "scp_codes"] + SUPERCLASSES].head()

**Expected output (real data):** about 21,799 records from about 18,869 patients; about 21,388 keep at least one superclass.

## Step 1.3 - Table 4.1: records per superclass and split

In [ ]:
from preprocessing.splits import split_by_fold, check_no_patient_overlap
splits = split_by_fold(meta)
check_no_patient_overlap(meta, splits)
print("No patient appears in more than one split.")

rows = {}
for name, idx in [("Training (folds 1-8)", splits["train"]), ("Validation (fold 9)", splits["val"]),
                  ("Test (fold 10)", splits["test"])]:
    m = meta.iloc[idx]
    rows[name] = {**{c: int(m[c].sum()) for c in SUPERCLASSES}, "Records": len(m), "Patients": m.patient_id.nunique()}
m = meta
rows["Total"] = {**{c: int(m[c].sum()) for c in SUPERCLASSES}, "Records": len(m), "Patients": m.patient_id.nunique()}
table41 = pd.DataFrame(rows).T
table41.to_csv(f"{RESULTS}/tables/table4_1_ptbxl_splits.csv")
table41

## Step 1.4 - Figure 4.1: label counts and co-occurrence

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
meta[SUPERCLASSES].sum().plot.bar(ax=ax[0], color="#1f6f8b"); ax[0].set_title("(a) Records per superclass")
meta["n_superclasses"].value_counts().sort_index().plot.bar(ax=ax[1], color="#1f6f8b")
ax[1].set_title("(b) Superclasses per record"); ax[1].set_xlabel("number of superclasses")
co = meta[SUPERCLASSES].T.dot(meta[SUPERCLASSES])
im = ax[2].imshow(co, cmap="Blues"); ax[2].set_xticks(range(5), SUPERCLASSES); ax[2].set_yticks(range(5), SUPERCLASSES)
for i in range(5):
    for j in range(5):
        ax[2].text(j, i, int(co.iloc[i, j]), ha="center", va="center", fontsize=8)
ax[2].set_title("(c) Co-occurrence")
plt.tight_layout(); plt.savefig(f"{RESULTS}/figures/fig4_1_ptbxl_labels.png", dpi=300); plt.show()
print("Records per patient:", meta.groupby("patient_id").size().describe().round(2).to_dict())

## Step 1.5 - Look at one ECG

In [ ]:
import wfdb
from preprocessing.ptbxl import LEADS
rec = meta.iloc[0]
sig, fields = wfdb.rdsamp(os.path.join(PTBXL, rec.filename_lr))
print("shape:", sig.shape, " sampling rate:", fields["fs"], "Hz  leads:", fields["sig_name"])
fig, axes = plt.subplots(12, 1, figsize=(12, 14), sharex=True)
t = np.arange(sig.shape[0]) / fields["fs"]
for l in range(12):
    axes[l].plot(t, sig[:, l], lw=0.8, color="k"); axes[l].set_ylabel(LEADS[l], rotation=0, labelpad=15)
axes[-1].set_xlabel("time (s)"); fig.suptitle(f"ecg_id {rec.name}: {rec.scp_codes}")
plt.tight_layout(); plt.savefig(f"{RESULTS}/figures/example_raw_ecg.png", dpi=200); plt.show()

**Done.** Copy Table 4.1 (`results/tables/table4_1_ptbxl_splits.csv`) and Figure 4.1 into Chapter 4, Section 4.2. Next: notebook 02.